In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

ROOT_DIR = Path.cwd().parent

BOUNDARY_DIR = ROOT_DIR / "data" / "raw" / "행정구역경계_2025"

# SHP 파일 자동 검색
shp_files = list(BOUNDARY_DIR.glob("*.shp"))

print("SHP 파일 수:", len(shp_files))

for file in shp_files:
    print(file.name)

SHP 파일 수: 1
bnd_sigungu_00_2025_2Q.shp


In [3]:
boundary_df = gpd.read_file(shp_files[0])

print("데이터 크기:", boundary_df.shape)
print("좌표계:", boundary_df.crs)

display(boundary_df.head())

데이터 크기: (252, 4)
좌표계: EPSG:5179


,BASE_DATE,SIGUNGU_CD,SIGUNGU_NM,geometry
0,20250630,11010,종로구,"POLYGON ((953683.828 1959209.871, 953665.283 1..."
1,20250630,11020,중구,"POLYGON ((957890.386 1952616.746, 957909.908 1..."
2,20250630,11030,용산구,"POLYGON ((953114.222 1950746.999, 953114.218 1..."
3,20250630,11040,성동구,"POLYGON ((959381.838 1952723.894, 959416.042 1..."
4,20250630,11050,광진구,"POLYGON ((964825.058 1952633.25, 964875.581 19..."


In [4]:
print("전체 행정구역 수:", len(boundary_df))

print("\n시군구 이름 결측치:")
print(boundary_df["SIGUNGU_NM"].isna().sum())

print("\n시군구 코드 중복:")
print(boundary_df["SIGUNGU_CD"].duplicated().sum())

print("\n행정구역 이름:")
display(
    boundary_df[["SIGUNGU_CD", "SIGUNGU_NM"]]
    .sort_values("SIGUNGU_CD")
    .reset_index(drop=True)
)

전체 행정구역 수: 252

시군구 이름 결측치:
0

시군구 코드 중복:
0

행정구역 이름:


,SIGUNGU_CD,SIGUNGU_NM
0,11010,종로구
1,11020,중구
2,11030,용산구
3,11040,성동구
4,11050,광진구
...,...,...
247,38580,함양군
248,38590,거창군
249,38600,합천군
250,39010,제주시


In [5]:
boundary_df["시도코드"] = (
    boundary_df["SIGUNGU_CD"]
    .astype(str)
    .str[:2]
)

print("시도코드별 행정구역 수")
display(
    boundary_df.groupby("시도코드")
    .agg(
        행정구역수=("SIGUNGU_NM", "count"),
        행정구역목록=("SIGUNGU_NM", lambda x: ", ".join(x))
    )
    .reset_index()
)

시도코드별 행정구역 수


,시도코드,행정구역수,행정구역목록
0,11,25,"종로구, 중구, 용산구, 성동구, 광진구, 동대문구, 중랑구, 성북구, 강북구, 도..."
1,21,16,"중구, 서구, 동구, 영도구, 부산진구, 동래구, 남구, 북구, 해운대구, 사하구,..."
2,22,9,"중구, 동구, 서구, 남구, 북구, 수성구, 달서구, 달성군, 군위군"
3,23,10,"중구, 동구, 연수구, 남동구, 부평구, 계양구, 서구, 미추홀구, 강화군, 옹진군"
4,24,5,"동구, 서구, 남구, 북구, 광산구"
5,25,5,"동구, 중구, 서구, 유성구, 대덕구"
6,26,5,"중구, 남구, 동구, 북구, 울주군"
7,29,1,세종시
8,31,44,"수원시 장안구, 수원시 권선구, 수원시 팔달구, 수원시 영통구, 성남시 수정구, 성..."
9,32,18,"춘천시, 원주시, 강릉시, 동해시, 태백시, 속초시, 삼척시, 홍천군, 횡성군, 영..."


In [6]:
# 행정구역명에 공백이 포함된 지역 확인
display(
    boundary_df[
        boundary_df["SIGUNGU_NM"].str.contains(" ", na=False)
    ][["SIGUNGU_CD", "SIGUNGU_NM"]]
    .sort_values("SIGUNGU_CD")
    .reset_index(drop=True)
)

,SIGUNGU_CD,SIGUNGU_NM
0,31011,수원시 장안구
1,31012,수원시 권선구
2,31013,수원시 팔달구
3,31014,수원시 영통구
4,31021,성남시 수정구
5,31022,성남시 중원구
6,31023,성남시 분당구
7,31041,안양시 만안구
8,31042,안양시 동안구
9,31051,부천시 원미구


In [7]:
# 원본 데이터 보호를 위해 복사
boundary_work = boundary_df.copy()

# 시도코드 → 시도명 매핑
sido_code_map = {
    "11": "서울특별시",
    "21": "부산광역시",
    "22": "대구광역시",
    "23": "인천광역시",
    "24": "광주광역시",
    "25": "대전광역시",
    "26": "울산광역시",
    "29": "세종특별자치시",
    "31": "경기도",
    "32": "강원특별자치도",
    "33": "충청북도",
    "34": "충청남도",
    "35": "전북특별자치도",
    "36": "전라남도",
    "37": "경상북도",
    "38": "경상남도",
    "39": "제주특별자치도"
}

boundary_work["시도코드"] = (
    boundary_work["SIGUNGU_CD"]
    .astype(str)
    .str[:2]
)

boundary_work["시도"] = (
    boundary_work["시도코드"]
    .map(sido_code_map)
)

# 일반구는 상위 시 이름으로 변경
boundary_work["시군구"] = (
    boundary_work["SIGUNGU_NM"]
    .str.split()
    .str[0]
)

display(
    boundary_work[["SIGUNGU_NM", "시도", "시군구"]]
    .head(20)
)

,SIGUNGU_NM,시도,시군구
0,종로구,서울특별시,종로구
1,중구,서울특별시,중구
2,용산구,서울특별시,용산구
3,성동구,서울특별시,성동구
4,광진구,서울특별시,광진구
5,동대문구,서울특별시,동대문구
6,중랑구,서울특별시,중랑구
7,성북구,서울특별시,성북구
8,강북구,서울특별시,강북구
9,도봉구,서울특별시,도봉구


In [8]:
boundary_merged = boundary_work.dissolve(
    by=["시도", "시군구"],
    as_index=False
)

print("통합 전 행정구역 수:", len(boundary_work))
print("통합 후 행정구역 수:", len(boundary_merged))

display(
    boundary_merged[["시도", "시군구"]]
    .sort_values(["시도", "시군구"])
    .reset_index(drop=True)
)

통합 전 행정구역 수: 252
통합 후 행정구역 수: 229


,시도,시군구
0,강원특별자치도,강릉시
1,강원특별자치도,고성군
2,강원특별자치도,동해시
3,강원특별자치도,삼척시
4,강원특별자치도,속초시
...,...,...
224,충청북도,제천시
225,충청북도,증평군
226,충청북도,진천군
227,충청북도,청주시


In [9]:
boundary_merged["시도"] = boundary_merged["시도"].replace({
    "광주광역시": "전남광주통합특별시",
    "전라남도": "전남광주통합특별시"
})

print("통합 후 지역 수:", len(boundary_merged))

print("중복 행정구역:")
display(
    boundary_merged[
        boundary_merged.duplicated(
            subset=["시도", "시군구"],
            keep=False
        )
    ][["시도", "시군구"]]
)

통합 후 지역 수: 229
중복 행정구역:


,시도,시군구


In [12]:
from pathlib import Path
import pandas as pd

ROOT_DIR = Path.cwd().parent
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

print("프로젝트 경로:", ROOT_DIR)
print("전처리 데이터 경로:", PROCESSED_DIR)

# 기존 고령인구 데이터와 비교
elderly_df = pd.read_csv(
    PROCESSED_DIR / "65세이상_고령인구_정제.csv"
)

elderly_df = elderly_df.rename(
    columns={"행정구역(시군구)별": "시군구"}
)

elderly_df["시도"] = elderly_df["시도"].replace({
    "광주광역시": "전남광주통합특별시",
    "전라남도": "전남광주통합특별시"
})

elderly_df["시군구"] = elderly_df["시군구"].replace({
    "세종특별자치시": "세종시",
    "창원시(통합)": "창원시"
})

# 지역 키 비교
shp_regions = set(
    zip(boundary_merged["시도"], boundary_merged["시군구"])
)

elderly_regions = set(
    zip(elderly_df["시도"], elderly_df["시군구"])
)

print("SHP에만 있는 지역:")
print(sorted(shp_regions - elderly_regions))

print("\n고령인구 데이터에만 있는 지역:")
print(sorted(elderly_regions - shp_regions))

print("\nSHP 지역 수:", len(shp_regions))
print("고령인구 지역 수:", len(elderly_regions))
print("공통 지역 수:", len(shp_regions & elderly_regions))

프로젝트 경로: c:\Users\expri\Desktop\SafeWeather-AI
전처리 데이터 경로: c:\Users\expri\Desktop\SafeWeather-AI\data\processed
SHP에만 있는 지역:
[]

고령인구 데이터에만 있는 지역:
[]

SHP 지역 수: 229
고령인구 지역 수: 229
공통 지역 수: 229


In [13]:
# 행정구역 내부의 대표 지점 계산
boundary_merged["대표점"] = (
    boundary_merged.geometry.representative_point()
)

# 대표점의 좌표 추출
boundary_merged["경도"] = boundary_merged["대표점"].x
boundary_merged["위도"] = boundary_merged["대표점"].y

# 좌표계 변환을 위해 대표점 GeoDataFrame 생성
region_points = gpd.GeoDataFrame(
    boundary_merged.drop(columns=["geometry", "대표점"]),
    geometry=boundary_merged["대표점"],
    crs=boundary_merged.crs
)

# EPSG:4326으로 변환 (위도·경도)
region_points = region_points.to_crs(epsg=4326)

region_points["경도"] = region_points.geometry.x
region_points["위도"] = region_points.geometry.y

display(
    region_points[
        ["시도", "시군구", "경도", "위도"]
    ].head(10)
)

,시도,시군구,경도,위도
0,강원특별자치도,강릉시,128.883209,37.709604
1,강원특별자치도,고성군,128.358361,38.401009
2,강원특별자치도,동해시,129.045397,37.515759
3,강원특별자치도,삼척시,129.173724,37.257882
4,강원특별자치도,속초시,128.515038,38.175014
5,강원특별자치도,양구군,127.988811,38.166345
6,강원특별자치도,양양군,128.622440,38.011551
7,강원특별자치도,영월군,128.455573,37.219057
8,강원특별자치도,원주시,127.899765,37.325059
9,강원특별자치도,인제군,128.254147,38.100548


In [14]:
print("전체 지역 수:", len(region_points))

print("\n결측치:")
print(region_points[["경도", "위도"]].isna().sum())

print("\n중복 지역:")
print(
    region_points.duplicated(
        subset=["시도", "시군구"]
    ).sum()
)

print("\n경도 범위:")
print(region_points["경도"].min(), "~", region_points["경도"].max())

print("\n위도 범위:")
print(region_points["위도"].min(), "~", region_points["위도"].max())

전체 지역 수: 229

결측치:
경도    0
위도    0
dtype: int64

중복 지역:
0

경도 범위:
124.673125177643 ~ 130.86011422839607

위도 범위:
33.33905798659285 ~ 38.40100919729494


In [15]:
output_path = PROCESSED_DIR / "시군구_대표좌표_정제.csv"

region_points[
    ["시도", "시군구", "경도", "위도"]
].to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)

저장 완료: c:\Users\expri\Desktop\SafeWeather-AI\data\processed\시군구_대표좌표_정제.csv
